In [1]:
import openmdao.api as om
import numpy as np
from pycycle.sofc_api import HeatConvection, HeatConvectionChannel, HeatConvectionElectrode, HeatConductionStructure, HeatConduction

+  
+  ==========================
+  thermo.base_thermo.chem_eq
+  ==========================
+  NL: Newton 0 ; 0.185589463 1
+  NL: Newton 1 ; 0.625416649 3.36989309
+  NL: Newton 2 ; 1.77484595 9.56329052
+  NL: Newton 3 ; 0.101422604 0.546489024
+  NL: Newton 4 ; 1307.53537 7045.31038
+  NL: Newton 5 ; 0.00232776883 0.01254257
+  NL: Newton 6 ; 2.10435662e-08 1.13387721e-07
+  NL: Newton Converged
+  
+  ==========================
+  thermo.base_thermo.chem_eq
+  ==========================
+  NL: Newton 0 ; 3.36265871e-10 1
+  NL: Newton Converged
+  
+  ==========================
+  thermo.base_thermo.chem_eq
+  ==========================
+  NL: Newton 0 ; 1.04215364e-10 1
+  NL: Newton Converged
+  
+  ==========================
+  thermo.base_thermo.chem_eq
+  ==========================
+  NL: Newton 0 ; 5.87234717e-12 1
+  NL: Newton Converged


### Simple Convection Test


In [2]:
model = om.Group()
model.add_subsystem('anode_channel', HeatConvectionChannel())

prob = om.Problem(model)
prob.setup(force_alloc_complex=True)

prob.set_val('anode_channel.W_in', 5)
prob.set_val('anode_channel.W_out', 5)
prob.set_val('anode_channel.h_in', 100)
prob.set_val('anode_channel.h_out', 300)
prob.run_model()



In [4]:
prob.get_val('anode_channel.Q_conv_ch', units='W', get_remote=True)

array([1000.])

##### Electrode

In [18]:
model = om.Group()
model.add_subsystem('anode', HeatConvectionElectrode(electrode = 'anode', structure = 'PEN'))
prob = om.Problem(model)
prob.setup(force_alloc_complex=True)


# Channel geometry  (identical for anode and cathode)
delta_an  = 35e-6    # m  anode layer thickness
delta_el  = 90e-6    # m  electrolyte thickness
delta_cat = 35e-6    # m  cathode layer thickness
delta_pen = delta_an + delta_el + delta_cat   # 160 um

thickness_IC = 850e-6   # m  interconnect thickness

# Cell planform
length_active  = 0.09    # m  active cell length  (= length_ESC)
length_total   = 0.162   # m
length_passive = length_total - length_active   # 0.072 m
width_total    = 0.142   # m  (= width_total_ESC, used as conduction cross-section height)

N_seg_active  = 10
N_seg_passive = 8

# Channel geometry  (identical for anode and cathode)
N_channels   = 24
w_ch         = 0.0042    # m  channel width
h_ch         = 1e-3      # m  channel height
d_hyd        = 2*w_ch*h_ch / (w_ch + h_ch)   # hydraulic diameter ≈ 1.615e-3 m
A_cross      = w_ch * h_ch * N_channels       # total flow cross-section per electrode

Nu = 4.9   # Nusselt number (both electrodes)


# Per-segment convection areas  [passive zone]
seg_len      = length_passive / N_seg_passive                  # 0.009 m
A_pen        = N_channels * w_ch * seg_len
A_ic_side    = N_channels * 2 * h_ch * seg_len

prob.set_val('anode.T_channel_in',    790)
prob.set_val('anode.T_channel_out',   795)
prob.set_val('anode.T_struc_out',     1000)
prob.set_val('anode.Nu',      4.9)
prob.set_val('anode.d_hyd', d_hyd, units='m')
prob.set_val('anode.A', A_cross, units='m**2')
prob.set_val('anode.lambda', 0.1805, units='W/m/K')  # thermal conductivity of Hydrogen
prob.set_val('anode.A_ic_side', A_ic_side, units='m**2')
prob.run_model()

In [ ]:
print(prob.get_val('anode.Q_conv_', units='W', get_remote=True))
print(prob.get_val('anode.T_bulk', units='K', get_remote=True))

[11.3138844]
[792.5]


In [ ]:
print(d_hyd)